In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')


In [8]:
df = pd.read_csv('data.csv')

In [9]:
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


## Dataset Features

- **id**: Unique ID for the customer  
- **Gender**: Gender of the customer  
- **Age**: Age of the customer  
- **Driving_License**:  
  - 0 → Customer does not have DL  
  - 1 → Customer already has DL  
- **Region_Code**: Unique code for the region of the customer  
- **Previously_Insured**:  
  - 1 → Customer already has Vehicle Insurance  
  - 0 → Customer doesn't have Vehicle Insurance  
- **Vehicle_Age**: Age of the Vehicle  
- **Vehicle_Damage**:  
  - 1 → Customer got his/her vehicle damaged in the past  
  - 0 → Customer didn't get his/her vehicle damaged in the past  
- **Annual_Premium**: The amount customer needs to pay as premium in the year  
- **Policy_Sales_Channel**: Anonymized Code for the channel of outreaching to the customer (Different Agents, Over Mail, Over Phone, In Person, etc.)  
- **Vintage**: Number of Days, Customer has been associated with the company  
- **Response**:  
  - 1 → Customer is interested  
  - 0 → Customer is not interested  


##  1. Exploratory Data Analysis (EDA)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


In [11]:
df.shape

(381109, 12)

In [12]:
df.describe()

,id,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response
count,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000
mean,190555.000000,38.822584,0.997869,26.388807,0.458210,30564.389581,112.034295,154.347397,0.122563
std,110016.836208,15.511611,0.046110,13.229888,0.498251,17213.155057,54.203995,83.671304,0.327936
min,1.000000,20.000000,0.000000,0.000000,0.000000,2630.000000,1.000000,10.000000,0.000000
25%,95278.000000,25.000000,1.000000,15.000000,0.000000,24405.000000,29.000000,82.000000,0.000000
50%,190555.000000,36.000000,1.000000,28.000000,0.000000,31669.000000,133.000000,154.000000,0.000000
75%,285832.000000,49.000000,1.000000,35.000000,1.000000,39400.000000,152.000000,227.000000,0.000000
max,381109.000000,85.000000,1.000000,52.000000,1.000000,540165.000000,163.000000,299.000000,1.000000


In [13]:
df.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

In [14]:
df.nunique()

id                      381109
Gender                       2
Age                         66
Driving_License              2
Region_Code                 53
Previously_Insured           2
Vehicle_Age                  3
Vehicle_Damage               2
Annual_Premium           48838
Policy_Sales_Channel       155
Vintage                    290
Response                     2
dtype: int64

In [15]:
df['Gender'].value_counts()


Gender
Male      206089
Female    175020
Name: count, dtype: int64

In [16]:
df['Driving_License'].value_counts()

Driving_License
1    380297
0       812
Name: count, dtype: int64

In [17]:
df['Previously_Insured'].value_counts()

Previously_Insured
0    206481
1    174628
Name: count, dtype: int64

In [18]:
df['Vehicle_Age'].value_counts()

Vehicle_Age
1-2 Year     200316
< 1 Year     164786
> 2 Years     16007
Name: count, dtype: int64

In [19]:
df['Vehicle_Damage'].value_counts()

Vehicle_Damage
Yes    192413
No     188696
Name: count, dtype: int64

In [20]:
df['Response'].value_counts()

Response
0    334399
1     46710
Name: count, dtype: int64

## Feature Engineering 

In [21]:
# Customer Tenure Bucket (Vintage ko categories mein baanto)

df['Tenure_Years'] = df['Vintage'] / 365
df['Tenure_Bucket'] = pd.cut(df['Tenure_Years'], bins=[0, 1, 3, 5, 10], labels=[0, 1, 2, 3])
df['Tenure_Bucket'] = df['Tenure_Bucket'].astype(int)


In [22]:
# Premium to Age Ratio (Pehle tha, lekin isko log transform karte hain)

df['Premium_Per_Age'] = np.log1p(df['Annual_Premium'] / (df['Age'] + 1))

In [23]:
# Is the customer a high-risk profile? (Purani gadi + high premium + no previous insurance)
df['High_Risk_Profile'] = ((df['Vehicle_Age'] == '> 2 Years') &
                           (df['Annual_Premium'] > df['Annual_Premium'].median()) &
                           (df['Previously_Insured'] == 0)).astype(int)


In [24]:
#  Interaction between Vehicle Damage and Vehicle Age
df['Damage_Old_Vehicle'] = ((df['Vehicle_Damage'] == 'Yes') &
                            (df['Vehicle_Age'].isin(['1-2 Year', '> 2 Years']))).astype(int)

In [25]:
# Premium Elasticity (Premium kitna extreme hai average se)
df['Premium_Elasticity'] = np.abs(df['Annual_Premium'] - df['Annual_Premium'].mean()) / df['Annual_Premium'].std()

In [26]:
# Encode karo saare categorical columns ko
df['Vehicle_Age_Encoded'] = df['Vehicle_Age'].map({'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2})
df['Vehicle_Damage_Encoded'] = df['Vehicle_Damage'].apply(lambda x: 1 if x == 'Yes' else 0)
df['Gender_Encoded'] = df['Gender'].map({'Male': 1, 'Female': 0})

In [27]:
# Frequency Encoding for high-cardinality columns
region_freq = df['Region_Code'].value_counts().to_dict()
df['Region_Code_Freq'] = df['Region_Code'].map(region_freq)
channel_freq = df['Policy_Sales_Channel'].value_counts().to_dict()
df['Policy_Sales_Channel_Freq'] = df['Policy_Sales_Channel'].map(channel_freq)

In [28]:
#  Bekar columns hatao
df.drop(['id', 'Driving_License', 'Gender', 'Vehicle_Age', 'Vehicle_Damage',
         'Region_Code', 'Policy_Sales_Channel', 'Tenure_Years'], axis=1, inplace=True)

In [29]:
#  Outliers ko handle karo (Cap karo)
upper_limit = df['Annual_Premium'].quantile(0.99)
df['Annual_Premium'] = np.where(df['Annual_Premium'] > upper_limit, upper_limit, df['Annual_Premium'])

##  Data Preparation & Splitting

In [30]:
X = df.drop('Response', axis=1)
y = df['Response']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Feature Scaling

In [31]:
cols_to_scale = ['Age', 'Annual_Premium', 'Vintage', 'Premium_Per_Age', 'Premium_Elasticity']
scaler = StandardScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

## Handling Class Imbalance

In [32]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

## Model Training & Hyperparameter Tuning

In [33]:
xgb = XGBClassifier(
    random_state=42, 
    eval_metric='logloss', 
    use_label_encoder=False,
)

In [34]:
param_grid = {
    'n_estimators': [150, 200],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.7, 0.8]
}

grid_search = GridSearchCV(xgb, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_search.fit(X_train_res, y_train_res)

print("\n===== BEST PARAMETERS (Auto-Tuned) =====")
print(grid_search.best_params_)


Fitting 3 folds for each of 48 candidates, totalling 144 fits

===== BEST PARAMETERS (Auto-Tuned) =====
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'subsample': 0.9}


## Model Evaluation

In [35]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

In [36]:
print("\n===== FINAL RESULTS (MASTER UPGRADE) =====")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\n===== DETAILED CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred))


===== FINAL RESULTS (MASTER UPGRADE) =====
Accuracy: 0.7982
F1-Score: 0.4251
AUC-ROC: 0.8444

===== DETAILED CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.94      0.82      0.88     66880
           1       0.33      0.61      0.43      9342

    accuracy                           0.80     76222
   macro avg       0.63      0.72      0.65     76222
weighted avg       0.86      0.80      0.82     76222



In [37]:
# 13. Feature Importance (Kaunse naye features kaam aaye?)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n===== TOP 10 FEATURES (Ismein naye features bhi aayenge) =====")
print(feature_importance.head(10))


===== TOP 10 FEATURES (Ismein naye features bhi aayenge) =====
                      Feature  Importance
1          Previously_Insured    0.969585
10     Vehicle_Damage_Encoded    0.014205
0                         Age    0.005806
7          Damage_Old_Vehicle    0.004627
9         Vehicle_Age_Encoded    0.001357
13  Policy_Sales_Channel_Freq    0.000899
2              Annual_Premium    0.000626
8          Premium_Elasticity    0.000598
5             Premium_Per_Age    0.000585
6           High_Risk_Profile    0.000509


In [39]:
# save model

import pickle

filename = 'rf_model.pkl'
pickle.dump(best_model, open(filename, 'wb'))

In [40]:
# loading back pickle file

rf_load = pickle.load(open(filename, 'rb'))